In [1]:
import sqlite3
import pandas as pd

RAW = "data/raw/"

# Which exh13 import column pairs with which FRED series
CATEGORY_MAP = {
    "Capital Goods":             ("Capital Goods",  "IR2"),
    "Automotive Vehicles, etc.": ("Automotive",     "IR3"),
    "Consumer Goods":            ("Consumer Goods", "IR4"),
}

In [2]:
census_raw = pd.read_excel(RAW + "exh13.xlsx", header=None)
census_raw.index = census_raw.index + 1   # index now equals the Excel row number
print("exh13 raw shape:", census_raw.shape)

fred_raw = {}
for _, series in CATEGORY_MAP.values():
    fred_raw[series] = pd.read_csv(RAW + f"{series}.csv")
    print(f"{series} raw shape:", fred_raw[series].shape)

exh13 raw shape: (70, 10)
IR2 raw shape: (561, 2)
IR3 raw shape: (543, 2)
IR4 raw shape: (531, 2)


In [3]:
header = census_raw.loc[6]                       # Excel row 6 holds the category names
imports_start = census_raw.index[census_raw[1] == "Imports"][0]
block = census_raw.loc[imports_start + 1:].copy()

label = block[0].astype(str).str.strip()
block["year"] = label.where(label.str.fullmatch(r"\d{4}")).ffill()

is_month = label.str.replace(" (R)", "", regex=False).isin(
    pd.date_range("2025-01-01", periods=12, freq="MS").strftime("%B"))
block = block[is_month & block[1].notna()].copy()   # drops summary, year, placeholder and footnote rows

block["month"] = pd.to_datetime(
    label[block.index].str.replace(" (R)", "", regex=False) + " " + block["year"],
    format="%B %Y").dt.strftime("%Y-%m-%d")         # same format FRED uses
block["excel_row"] = block.index

cols = {header[header == name].index[0]: short for name, (short, _) in CATEGORY_MAP.items()}
census_long = (block.rename(columns=cols)
               .melt(id_vars=["month", "excel_row"], value_vars=list(cols.values()),
                     var_name="category", value_name="import_value_musd"))
census_long["import_value_musd"] = census_long["import_value_musd"].astype(float)
print("census_long shape:", census_long.shape)
census_long.head()

census_long shape: (57, 4)


,month,excel_row,category,import_value_musd
0,2025-01-01,41,Capital Goods,83745.0
1,2025-02-01,42,Capital Goods,77141.0
2,2025-03-01,43,Capital Goods,93464.0
3,2025-04-01,44,Capital Goods,89168.0
4,2025-05-01,45,Capital Goods,90846.0


In [4]:
frames = []
for short, series in CATEGORY_MAP.values():
    f = fred_raw[series].rename(columns={series: "price_index"})
    f["series_id"] = series
    f["category"] = short
    f["csv_line"] = f.index + 2                     # +1 for header, +1 for 1-based lines
    frames.append(f)
fred_long = pd.concat(frames, ignore_index=True)
print("fred_long shape:", fred_long.shape)
print("blank price_index cells:", fred_long["price_index"].isna().sum())

fred_long shape: (1635, 5)
blank price_index cells: 187


In [5]:
con = sqlite3.connect(":memory:")
census_long.to_sql("census_imports", con, index=False)
fred_long.to_sql("fred_price_index", con, index=False)

JOIN_SQL = """
SELECT c.month,
       c.category,
       c.import_value_musd,
       f.series_id,
       f.price_index
FROM census_imports AS c
INNER JOIN fred_price_index AS f
        ON c.month    = f.observation_date
       AND c.category = f.category
ORDER BY c.category, c.month;
"""
df = pd.read_sql(JOIN_SQL, con)

In [6]:
df.shape

(57, 5)

In [7]:
df.sample(10, random_state=42)

,month,category,import_value_musd,series_id,price_index
0,2025-01-01,Automotive,35502.0,IR3,121.8
5,2025-06-01,Automotive,34926.0,IR3,121.0
30,2025-12-01,Capital Goods,111479.0,IR2,95.1
13,2026-02-01,Automotive,29399.0,IR3,120.4
34,2026-04-01,Capital Goods,123433.0,IR2,98.4
55,2026-06-01,Consumer Goods,57282.0,IR4,112.2
27,2025-09-01,Capital Goods,91360.0,IR2,94.9
31,2026-01-01,Capital Goods,102117.0,IR2,95.7
45,2025-08-01,Consumer Goods,56298.0,IR4,110.1
12,2026-01-01,Automotive,28643.0,IR3,120.3


In [8]:
before = len(census_long)
after = len(df)
summary = pd.DataFrame({
    "Metric": ["Row count before join (Census)", "Row count after join",
               "Rows lost", "Match rate"],
    "Value": [before, after, before - after, f"{after / before:.0%}"],
})
summary

,Metric,Value
0,Row count before join (Census),57
1,Row count after join,57
2,Rows lost,0
3,Match rate,100%


In [9]:
unmatched_fred = pd.read_sql("""
SELECT f.observation_date, f.series_id, f.price_index
FROM fred_price_index AS f
LEFT JOIN census_imports AS c
       ON c.month = f.observation_date AND c.category = f.category
WHERE c.month IS NULL AND f.observation_date >= '2025-01-01'
ORDER BY f.observation_date, f.series_id;
""", con)
unmatched_fred

,observation_date,series_id,price_index
0,2026-08-01,IR2,101.9
1,2026-08-01,IR3,121.2
2,2026-08-01,IR4,112.7


In [10]:
pd.DataFrame({
    "before": census_long["category"].value_counts(),
    "after": df["category"].value_counts(),
})

,before,after
category,,
Automotive,19,19
Capital Goods,19,19
Consumer Goods,19,19


In [11]:
pd.concat({
    "before": census_long.groupby("category")["import_value_musd"].describe()[["count", "mean", "min", "max"]],
    "after": df.groupby("category")["import_value_musd"].describe()[["count", "mean", "min", "max"]],
}, axis=1)

before                                   after                 \
                count           mean      min       max count           mean   
category                                                                       
Automotive       19.0   34839.315789  28643.0   43040.0  19.0   34839.315789   
Capital Goods    19.0  104354.368421  77141.0  148539.0  19.0  104354.368421   
Consumer Goods   19.0   62252.526316  48880.0   99686.0  19.0   62252.526316   

                                   
                    min       max  
category                           
Automotive      28643.0   43040.0  
Capital Goods   77141.0  148539.0  
Consumer Goods  48880.0   99686.0

In [12]:
df[df["price_index"].isna()]

,month,category,import_value_musd,series_id,price_index
9,2025-10-01,Automotive,34763.0,IR3,NaN
28,2025-10-01,Capital Goods,102897.0,IR2,NaN
47,2025-10-01,Consumer Goods,59835.0,IR4,NaN


In [13]:
blank_rows = fred_long[(fred_long["price_index"].isna()) &
                       (fred_long["observation_date"] >= "2025-01-01")]
blank_rows[["series_id", "csv_line", "observation_date", "price_index"]]

,series_id,csv_line,observation_date,price_index
550,IR2,552,2025-10-01,NaN
1093,IR3,534,2025-10-01,NaN
1624,IR4,522,2025-10-01,NaN
